# Synthetic generation pipeline — step by step

Walks through `synthetic/generator.py`'s pipeline one stage at a time — time array with gaps, flat baseline, noise injection, transit injection — so each module's effect is visible in isolation before looking at the final candidates.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
import yaml

from synthetic.generator import make_base_flux, make_time_array
from synthetic.noise_models import add_gaussian_noise, add_red_noise
from synthetic.transit_injector import inject_secondary_eclipse, inject_transit

with open("../synthetic/config.yaml") as f:
    config = yaml.safe_load(f)
gen = config["generation"]

## Stage 1 — time array with simulated data gaps

~2% of cadences are randomly dropped to mimic TESS momentum-dump / downlink gaps.

In [ ]:
time = make_time_array(
    n_points=gen["n_points"], time_span_days=gen["time_span_days"],
    cadence_minutes=gen["cadence_minutes"], seed=1,
)
gaps = np.diff(time)
cadence_days = gen["cadence_minutes"] / 1440.0

plt.figure(figsize=(9, 2.5))
plt.plot(time[:-1], gaps, ".", markersize=2)
plt.axhline(cadence_days, color="gray", linestyle="--", linewidth=0.5, label="expected cadence")
plt.ylabel("Δt (days)")
plt.xlabel("time (BTJD)")
plt.title(f"time spacing — {len(time)} points after gap removal")
plt.legend()
plt.tight_layout()
plt.show()

## Stage 2 — flat baseline flux

In [ ]:
flux_flat = make_base_flux(len(time))
print("all ones:", np.all(flux_flat == 1.0), " length:", len(flux_flat))

## Stage 3 — noise injection

Gaussian (white) vs. red (AR(1)-correlated) noise at the same amplitude, side by side.

In [ ]:
flux_gauss = add_gaussian_noise(flux_flat, sigma=0.003, seed=42)
flux_red = add_red_noise(flux_flat, sigma=0.003, correlation=0.6, seed=42)

window = slice(0, 2000)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, sharey=True)
axes[0].plot(time[window], flux_gauss[window], linewidth=0.6, color="steelblue")
axes[0].set_title("Gaussian (white) noise")
axes[1].plot(time[window], flux_red[window], linewidth=0.6, color="firebrick")
axes[1].set_title("Red (correlated) noise")
axes[1].set_xlabel("time (BTJD)")
for ax in axes:
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
fig.tight_layout()
plt.show()

## Stage 4 — transit injection

Box-shaped (exoplanet-like) vs. V-shaped with a secondary eclipse (eclipsing-binary-like), on otherwise clean flux so the shape is unambiguous.

In [ ]:
flux_box = flux_flat.copy()
inject_transit(flux_box, time, period_days=3.42, depth=0.013, duration_days=0.16, v_shape=False)

flux_v = flux_flat.copy()
inject_transit(flux_v, time, period_days=1.87, depth=0.18, duration_days=0.08, v_shape=True)
inject_secondary_eclipse(flux_v, time, period_days=1.87, secondary_depth=0.09, duration_days=0.08)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mask_a = time < 7
axes[0].plot(time[mask_a], flux_box[mask_a], linewidth=0.8, color="steelblue")
axes[0].set_title("Box transit (exoplanet_like)\nperiod=3.42d, depth=1.3%")
axes[0].set_xlabel("time (BTJD)")
axes[0].set_ylabel("flux")

mask_b = time < 4
axes[1].plot(time[mask_b], flux_v[mask_b], linewidth=0.8, color="firebrick")
axes[1].set_title("V-shape + secondary eclipse (eclipsing_binary_like)\nperiod=1.87d, depth=18%")
axes[1].set_xlabel("time (BTJD)")

for ax in axes:
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")
fig.tight_layout()
plt.show()

## Final candidates, side by side

The actual `synthetic/cases/*.csv` output — noise and transit injection combined, matching exactly what `generate_all_cases()` writes to disk.

In [ ]:
import pandas as pd

cases_dir = "../synthetic/cases"
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
colors = ["steelblue", "firebrick", "seagreen"]

for ax, case_name, color in zip(axes, ["candidate_a", "candidate_b", "candidate_c"], colors):
    csv_path = os.path.join(cases_dir, f"{case_name}.csv")
    if not os.path.exists(csv_path):
        from synthetic.generator import generate_all_cases
        generate_all_cases("../synthetic/config.yaml", cases_dir)
    df = pd.read_csv(csv_path)
    ax.plot(df["time"], df["flux"], ".", markersize=1, color=color)
    ax.set_title(case_name)
    ax.set_ylabel("flux")
    ax.axhline(1.0, color="gray", linewidth=0.5, linestyle="--")

axes[-1].set_xlabel("time (BTJD)")
fig.tight_layout()
plt.show()